# Taller 7 - CNN para clasificar pósters por género

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
import random
import numpy as np
import keras

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Keras: {keras.__version__}")

In [ ]:
import kagglehub
from pathlib import Path

path = Path(
    kagglehub.dataset_download("zulkarnainsaurav/four-genre-movie-poster-images")
)
print("Ruta descargada:", path)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from keras import layers
import warnings

warnings.filterwarnings("ignore")

## 1) Exploración del dataset

In [ ]:
DATA_DIR = Path(path) / "four_genre_posters" / "four_genre_posters"

In [ ]:
print(DATA_DIR)
IMG_SIZE = (380, 562)

action_paths = sorted(DATA_DIR.joinpath("Action").glob("*.jpg"))
comedy_paths = sorted(DATA_DIR.joinpath("Comedy").glob("*.jpg"))
horror_paths = sorted(DATA_DIR.joinpath("Horror").glob("*.jpg"))
romance_paths = sorted(DATA_DIR.joinpath("Romance").glob("*.jpg"))

n_action = len(action_paths)
n_comedy = len(comedy_paths)
n_horror = len(horror_paths)
n_romance = len(romance_paths)

print(f"Action: {n_action} imágenes")
print(f"Comedy: {n_comedy} imágenes")
print(f"Horror: {n_horror} imágenes")
print(f"Romance: {n_romance} imágenes")


_, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(
    ["Action", "Comedy", "Horror", "Romance"],
    [n_action, n_comedy, n_horror, n_romance],
    color=["tomato", "steelblue", "lightgreen", "orange"],
)
ax.set_title("Distribución de Clases")
ax.set_ylabel("Número de imágenes")
ax.bar_label(bars)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 6))
fig.suptitle("Muestras del Dataset:")

for i in range(5):
    img = Image.open(action_paths[i]).convert("RGB")
    axes[0, i].imshow(img)
    axes[0, i].set_title("Action")
    axes[0, i].axis("off")

for i in range(5):
    img = Image.open(comedy_paths[i]).convert("RGB")
    axes[1, i].imshow(img)
    axes[1, i].set_title("Comedy")
    axes[1, i].axis("off")

for i in range(5):
    img = Image.open(horror_paths[i]).convert("RGB")
    axes[2, i].imshow(img)
    axes[2, i].set_title("Horror")
    axes[2, i].axis("off")

for i in range(5):
    img = Image.open(romance_paths[i]).convert("RGB")
    axes[3, i].imshow(img)
    axes[3, i].set_title("Romance")
    axes[3, i].axis("off")

plt.tight_layout()
plt.show()

# 3. Preprocesamiento de datos

Aplicamos 3 tecnicas, normalizacion, estandarizacion y aumentacion de datos.

Junto a la carga donde dividimos los datos de manera estratificada.

Con 15% test, 15% validacion y 85% entrenamiento

In [ ]:
def load_dataset(
    action_paths, comedy_paths, horror_paths, romance_paths, img_size=(64, 64)
):
    imgs, labels = [], []
    for path in action_paths:
        img = Image.open(path).convert("RGB").resize(img_size)
        imgs.append(np.array(img, dtype=np.float32))
        labels.append([1, 0, 0, 0])
    for path in comedy_paths:
        img = Image.open(path).convert("RGB").resize(img_size)
        imgs.append(np.array(img, dtype=np.float32))
        labels.append([0, 1, 0, 0])
    for path in horror_paths:
        img = Image.open(path).convert("RGB").resize(img_size)
        imgs.append(np.array(img, dtype=np.float32))
        labels.append([0, 0, 1, 0])
    for path in romance_paths:
        img = Image.open(path).convert("RGB").resize(img_size)
        imgs.append(np.array(img, dtype=np.float32))
        labels.append([0, 0, 0, 1])
    return np.array(imgs), np.array(labels, dtype=np.float32)


X, y = load_dataset(action_paths, comedy_paths, horror_paths, romance_paths, IMG_SIZE)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp
)

print(f"Conjunto de entrenamiento: {len(X_train)} imágenes {len(X_train) / len(X):.2%}")
print(f"Conjunto de validación: {len(X_val)} imágenes {len(X_val) / len(X):.2%}")
print(f"Conjunto de prueba: {len(X_test)} imágenes {len(X_test) / len(X):.2%}")